# Harfi: Qwen3-1.7B v1 on spelling v2.0 and round 6

**Qwen3-1.7B** with exactly Llama v5's recipe (LoRA rank 32, alpha 64, lr 2e-4, 2 epochs, a checkpoint every 538 steps,
the last epoch's model kept) on the same `harfi-v6` export (144,986 examples, spelling v2.0 + round 6) that Qwen3-0.6B v2
and Llama 3.2 1B v5 trained on. Only the base model differs, so the three compare fairly: does a larger Qwen close
Qwen 0.6B's gap to Llama on text by other people (others test, −0.017; docs/EVAL.md, "Qwen v2 against v4")?
Qwen3-1.7B was trained once before, on the 8k-example v1 data only (dev 0.723 against Llama's 0.768).

Qwen3 trains in fp16 on the T4. Its thinking mode is turned off in the prompt, as for the other Qwen runs. Licence: Apache-2.0.

**Output:** `arabizi-qwen3-1.7b-v1-Q4_K_M.gguf` (about 1.1 GB, the level every shipped model uses) and `-Q5_K_M.gguf`
(about 1.26 GB, kept only if it fits the 1.5 GB memory budget and scores clearly better), their SHA-256 files and `run.json`
(with the exact prompt the engine needs). No Q8_0: at about 1.8 GB it cannot ship, and for Qwen 0.6B it was at most a
fraction of a point above Q4.

**How to run:**
1. Add Input -> the `harfi-v6` dataset (the same one as Qwen v2 and Llama v5), and only it.
2. Settings -> Accelerator -> **GPU T4** · **Internet on**.
3. **Save Version -> Save & Run All (Commit)**. Expected about 8 hours (Qwen v2 took 4.6 h; 1.7B was about 1.6 times
   Llama 1B's time in v1). Training stops by itself with a checkpoint after `BUDGET` minutes, before Kaggle's 12-hour limit.
4. If it finished: from the version's **Output** tab, **Download all**. If it printed **NOT FINISHED**: add this version's
   output as a second input (Add Input -> Your Work -> this notebook), keep `harfi-v6`, and Save & Run All again; it goes
   on from the last checkpoint and ends with the same weights as one uninterrupted run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU: the same effective batch (32) as v4 and v5

In [ ]:
REF = "feat/qwen3-1.7b-v1"  # this branch until it is merged (it has checkpoints and resuming), then "main"
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /kaggle/working/repo
!git clone --quiet --depth 1 --branch "$REF" https://github.com/Mustafa-Abbara/arabizi-keyboard /kaggle/working/repo
!git -C /kaggle/working/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /kaggle/working/repo/train/requirements.txt

In [ ]:
import glob
import json
import subprocess
import sys

exports = glob.glob("/kaggle/input/**/export.json", recursive=True)
assert len(exports) == 1, f"add exactly one dataset with export.json (found {exports})"
DATA = os.path.dirname(exports[0])
with open(exports[0]) as f:
    summary = json.load(f)
print(DATA, "|", summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
assert summary["rows_total"] == 144986, "this should be the harfi-v6 export (spelling v2.0 + round 6)"
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"

BASE, TAG, NAME = "qwen3-1.7b", "v1", "arabizi-qwen3-1.7b-v1"
# An earlier, unfinished part of this run, added as an input: go on from its last checkpoint.
earlier = sorted(glob.glob(f"/kaggle/input/**/*-{BASE}-lora-{TAG}/PARTIAL.json", recursive=True))
RESUME = os.path.dirname(earlier[-1]) if earlier else None
print("going on from", RESUME if RESUME else "nothing: a new run")

In [ ]:
# Llama v5's settings: rank 32, alpha 64, lr 2e-4, 2 epochs, a checkpoint every 538 steps (1/8 of an epoch).
# BUDGET: minutes of training before it stops with a checkpoint; setup, merging and the GGUF export take about 40 more,
# so 600 stays under Kaggle's 12 hours.
BUDGET = 600

print(f"=== {BASE} {TAG}" + (", resumed" if RESUME else ""), flush=True)
# check=True: a failed run stops the notebook instead of exporting nothing.
subprocess.run([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", BASE, "--data", DATA, "--out", OUT,
                "--tag", TAG, "--rank", "32", "--alpha", "64", "--epochs", "2",
                "--save-steps", "538", "--time-budget", str(BUDGET), *(["--resume", RESUME] if RESUME else [])],
               check=True)
run = max(glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}"))
if os.path.exists(f"{run}/PARTIAL.json"):
    with open(f"{run}/PARTIAL.json") as f:
        print("NOT FINISHED:", json.load(f), "-> add this version's output as an input and Save & Run All again")
else:
    subprocess.run(["bash", f"{REPO}/train/export_gguf.sh", f"{run}/merged", NAME, run, "Q4_K_M Q5_K_M"], check=True)
    missing = [q for q in ("Q4_K_M", "Q5_K_M") if not os.path.exists(f"{run}/{NAME}-{q}.gguf")]
    assert not missing, f"not exported: {missing}"
    subprocess.run(["rm", "-rf", f"{run}/merged", f"{run}/checkpoints"], check=True)

## Results and download

The best validation loss is only a first look (Qwen v2's and Llama v5's are in their run.json files; same data, but
different tokenizers, so compare accuracy, not loss). Accuracy on the dev sets and the real chats decides, scored against
Llama v5 and Qwen v2 with every answer in spelling v2.0. From the version's **Output** tab, use **Download all** (one zip).

In [ ]:
!rm -rf /kaggle/working/repo
for path in sorted(glob.glob(f"{OUT}/*-lora-*/run.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{r['run_id']:45} best validation loss {r['best_eval_loss']:.4f}  ({r['minutes']} min, {r['precision']})")
    for s in r["public_samples"]:
        print(f"  {s['arabizi']}\n    -> {s['output']}\n    want {s['reference']}")
!find "$OUT" -maxdepth 2 -type f -printf "%12s  %p\n"